# 🦙 Fine-Tuning Llama 3.1 & 3.2 with Unsloth and Hugging Face TRL

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/)
[![GitHub](https://img.shields.io/badge/GitHub-Repository-blue?logo=github)](https://github.com/unslothai/unsloth)

Welcome to the end-to-end LLM fine-tuning notebook. In this notebook, we use **Unsloth** and **Hugging Face TRL (`SFTTrainer`)** to fine-tune **Llama 3.2 (1B / 3B)** or **Llama 3.1 (8B)** using 4-bit QLoRA.

### 🌟 Key Features:
- ⚡ **2x to 5x faster training** compared to standard Hugging Face pipelines.
- 💾 **Up to 70% memory reduction**—comfortably train on free Google Colab T4 (16GB VRAM) or Kaggle GPUs.
- 🎯 **Full lifecycle support**: Data preparation, training loss curves, before-and-after inference, and multi-format exports (LoRA, 16-bit merged, and GGUF for Ollama / LM Studio).


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 1. Hardware & GPU Environment Check
Check your GPU specifications, compute capability, and whether native `bfloat16` is supported.


In [2]:
import torch

if torch.cuda.is_available():
    gpu_stats = torch.cuda.get_device_properties(0)
    start_gpu_memory = round(gpu_stats.total_memory / (1024 ** 3), 2)
    print(f"✅ GPU Detected: {gpu_stats.name}")
    print(f"📊 Total VRAM: {start_gpu_memory} GB")
    print(f"⚙️ Compute Capability: {gpu_stats.major}.{gpu_stats.minor}")
    print(f"🚀 bfloat16 Supported: {torch.cuda.is_bf16_supported()}")
else:
    print("⚠️ WARNING: No CUDA GPU detected. Please change your runtime to a GPU (e.g. T4 or A100 in Colab).")


✅ GPU Detected: Tesla T4
📊 Total VRAM: 14.56 GB
⚙️ Compute Capability: 7.5
🚀 bfloat16 Supported: True


## 2. Install Dependencies
Unsloth provides pre-compiled PyTorch/CUDA wheels for Google Colab, Kaggle, and Linux environments.


In [3]:
# 1. Install correct versions
%pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
%pip install --no-deps "trl>=0.18.2,<0.25.0" "datasets>=3.4.1,<4.4.0" peft accelerate bitsandbytes transformers

# 2. Force restart the runtime to load new versions
import os
os.kill(os.getpid(), 9)

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-kizz1et3/unsloth_9c1a09a1680d437dbf4bab052aa1e027
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-kizz1et3/unsloth_9c1a09a1680d437dbf4bab052aa1e027
  Resolved https://github.com/unslothai/unsloth.git to commit 7c63bc8c4f18c1d0b0eec5e656ae20797893b500
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


## 3. Load Base Model & Tokenizer
We load the base model in **4-bit quantization** using `FastLanguageModel.from_pretrained`.

- **Target Model**: `unsloth/Llama-3.2-1B-Instruct` (fast, lightweight, runs locally without gated token barriers).
- `max_seq_length`: Sequence length for input tokens (default: `2048`, dynamically expandable).
- `load_in_4bit = True`: Enables 4-bit QLoRA, drastically reducing memory consumption.
- `dtype = None`: Automatically selects `bfloat16` for Ampere+ GPUs (A100, L4) or `float16` for Turing GPUs (T4).


In [23]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048  # Supports RoPE scaling internally up to 8192+
dtype = None           # Auto-detection: Float16 on T4, Bfloat16 on Ampere/Ada/Hopper
load_in_4bit = True    # 4-bit quantization enabled

# Available pre-quantized models:
# - "unsloth/Llama-3.2-1B-Instruct" (Default: Ultra-fast, lightweight, fits in ~4-6GB VRAM)
# - "unsloth/Llama-3.2-3B-Instruct" (Balanced: High quality on T4 GPU)
# - "unsloth/Meta-Llama-3.1-8B-Instruct" (Flagship open model, fits 16GB VRAM)

model_name = "unsloth/llama-3-8b-Instruct"

print(f"Loading {model_name}...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=model_name,
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
)
print("✅ Base model and tokenizer loaded successfully!")


Loading unsloth/llama-3-8b-Instruct...
==((====))==  Unsloth 2026.9.4: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-instruct-bnb-4bit as a legacy tokenizer.


✅ Base model and tokenizer loaded successfully!


## 4. Configure LoRA / QLoRA Adapters
We inject Low-Rank Adaptation (LoRA) weights into all linear projection modules.
- `r = 16`: Rank of the low-rank matrices.
- `target_modules`: Configured across all attention and MLP projection layers (`q_proj`, `k_proj`, `v_proj`, `o_proj`, `gate_proj`, `up_proj`, `down_proj`).
- `lora_alpha = 16`: LoRA scaling factor.
- `use_gradient_checkpointing = "unsloth"`: Unsloth's optimized checkpointing saves significant VRAM.


In [25]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,  # 0 is optimized by Unsloth for maximum speed
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

trainable_params, all_param = model.get_nb_trainable_parameters()
print(f"🎯 Trainable parameters: {trainable_params:,} / {all_param:,} ({100 * trainable_params / all_param:.2f}%)")


Unsloth 2026.9.4 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


🎯 Trainable parameters: 41,943,040 / 8,072,204,288 (0.52%)


## 5. Dataset Preparation & Prompt Formatting
Fine-tuning models requires structuring training pairs using the official **Llama-3 Instruct** template.

This notebook automatically supports:
1. **Multi-turn Chat Conversations** (like your custom dataset in `dataset/` with `[{"role": "system"|"user"|"assistant", "content": "..."}]`).
2. **Instruction-Tuning Pairs** (e.g. `instruction`, `input`, `output` format from Alpaca).

```text
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>

{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

{response}<|eot_id|>
```


In [26]:
import os
import glob
from datasets import load_dataset

EOS_TOKEN = tokenizer.eos_token  # Appended to mark sequence termination

def formatting_prompts_func(examples):
    # Case 1: Multi-turn chat conversation (e.g. Denmohor dataset in dataset/)
    if "messages" in examples:
        texts = []
        for msg_list in examples["messages"]:
            formatted = ""
            for idx, m in enumerate(msg_list):
                role = m.get("role", "")
                content = m.get("content", "")
                if idx == 0 and role == "system":
                    formatted += f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{content}<|eot_id|>"
                elif role == "user":
                    formatted += f"<|start_header_id|>user<|end_header_id|>\n\n{content}<|eot_id|>"
                elif role == "assistant":
                    formatted += f"<|start_header_id|>assistant<|end_header_id|>\n\n{content}<|eot_id|>"
            texts.append(formatted)
        return {"text": texts}

    # Case 2: Alpaca single-turn format
    llama_prompt = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\n{system_prompt}<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n{response}<|eot_id|>"""
    instructions = examples.get("instruction", [])
    inputs = examples.get("input", [""] * len(instructions))
    outputs = examples.get("output", [])
    texts = []

    for instruction, input_text, output in zip(instructions, inputs, outputs):
        full_instruction = f"{instruction}\nInput: {input_text}" if input_text else instruction
        text = llama_prompt.format(
            system_prompt="You are a helpful, respectful, and honest assistant.",
            instruction=full_instruction,
            response=output
        ) + EOS_TOKEN
        texts.append(text)
    return {"text": texts}

# Check for custom dataset in dataset/ directory
custom_path = None
if os.path.exists("dataset/custom_dataset.jsonl"):
    custom_path = "dataset/custom_dataset.jsonl"
elif os.path.isdir("dataset") and glob.glob("dataset/*.json"):
    from prepare_dataset import consolidate_dataset
    custom_path = consolidate_dataset("dataset")

if custom_path and os.path.exists(custom_path):
    print(f"Loading custom dataset from {custom_path}...")
    dataset = load_dataset("json", data_files=custom_path, split="train")
else:
    print("Loading default community dataset (yahma/alpaca-cleaned)...")
    dataset = load_dataset("yahma/alpaca-cleaned", split="train[:3000]")

dataset = dataset.map(formatting_prompts_func, batched=True)
print(f"✅ Prepared {len(dataset)} training examples!")
print("-" * 60)
print(dataset[0]["text"][:400] + "...")
print("-" * 60)


Loading default community dataset (yahma/alpaca-cleaned)...
✅ Prepared 3000 training examples!
------------------------------------------------------------
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

You are a helpful, respectful, and honest assistant.<|eot_id|><|start_header_id|>user<|end_header_id|>

Give three tips for staying healthy.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, a...
------------------------------------------------------------


### 💡 Custom Dataset Info
The cell above auto-detected your custom dataset from `dataset/`! If you add new `.json` files into `dataset/`, run `python prepare_dataset.py` or re-run the cell above to include them.


## 6. Baseline Inference (Before Fine-Tuning)
Let's run a test prompt against the base model before training so we can observe how fine-tuning alters its response behavior.


In [27]:
from transformers import TextStreamer

# Enable fast inference mode
FastLanguageModel.for_inference(model)

# Test prompt (customized for legal assistance assistant)
test_instruction = "আমার স্বামী আমাকে দেনমোহরের টাকা পরিশোধ করতে অস্বীকার করছে। আমি কি আইনি পদক্ষেপ নিতে পারি?"
prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nআপনি একজন বাংলাদেশি পারিবারিক আইন সহায়তা সহকারী। ব্যবহারকারীর সমস্যার ভিত্তিতে প্রাসঙ্গিক পরবর্তী প্রশ্ন করুন।<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{test_instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
text_streamer = TextStreamer(tokenizer)

print("--- Output BEFORE Fine-Tuning ---")
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=256, use_cache=True)


--- Output BEFORE Fine-Tuning ---
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

আপনি একজন বাংলাদেশি পারিবারিক আইন সহায়তা সহকারী। ব্যবহারকারীর সমস্যার ভিত্তিতে প্রাসঙ্গিক পরবর্তী প্রশ্ন করুন।<|eot_id|><|start_header_id|>user<|end_header_id|>

আমার স্বামী আমাকে দেনমোহরের টাকা পরিশোধ করতে অস্বীকার করছে। আমি কি আইনি পদক্ষেপ নিতে পারি?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Both `max_new_tokens` (=256) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


আপনার স্বামী দেনমোহরের টাকা পরিশোধ করতে অস্বীকার করছে, তাহলে আপনার জন্য কিছু আইনি পদক্ষেপ নিতে পারেন না, কিন্তু আপনার স্বামী এবং আপনার মধ্যে বিচারপত্র দ্বারা সমাধান করা যাবে। 

কিন্তু আপনার স্বামী দেনমোহরের টাকা পরিশোধ �


## 7. Fine-Tuning with Hugging Face TRL `SFTTrainer`
We configure `SFTTrainer` with memory-optimized hyperparameters:
- `per_device_train_batch_size = 2` & `gradient_accumulation_steps = 4` (effective batch size: 8).
- `optim = "adamw_8bit"` to reduce optimizer memory overhead.
- `learning_rate = 2e-4` with cosine/linear decay.
- `max_steps = 60` for a fast demonstration run (adjust to `100-500` or full epochs for real datasets).


In [8]:
!pip install --upgrade trl peft accelerate

  Using cached trl-1.13.0-py3-none-any.whl.metadata (12 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
Using cached trl-1.13.0-py3-none-any.whl (1.0 MB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: trl
    Found existing installation: trl 0.24.0
    Uninstalling trl-0.24.0:
      Successfully uninstalled trl-0.24.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.9.3 requires datasets!=4.0.*,!=4.1.0,<4.4.0,>=3.4.1, but you have datasets 5.0.1 which is incompatible.
unsloth-zoo 2026.9.3 requires trl!=0.19.0,<=0.24.0,>=0.18.2; sys_platform != "darwin" or platform_machine != "arm64", but you have trl 1.13.0 which is incompatible.


In [28]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=max_seq_length,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=5,
        max_steps=60,
        learning_rate=2e-4,
        fp16=not is_bfloat16_supported(),
        bf16=is_bfloat16_supported(),
        logging_steps=10,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=3407,
        output_dir="outputs",
        report_to="none",
    ),
)

Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/3000 [00:00<?, ? examples/s]

In [29]:
# Record memory stats and launch training
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(gpu_stats.total_memory / (1024 ** 3), 2)
print(f"🚀 Training starting on {gpu_stats.name} (Total VRAM: {start_gpu_memory} GB)...")

trainer_stats = trainer.train()

# Report peak memory & training runtime
used_memory = round(torch.cuda.max_memory_reserved() / (1024 ** 3), 2)
print(f"✅ Training completed in {trainer_stats.metrics['train_runtime']:.2f} seconds!")
print(f"📊 Peak reserved VRAM: {used_memory} GB ({round(used_memory / start_gpu_memory * 100, 1)}% of total)")


🚀 Training starting on Tesla T4 (Total VRAM: 14.56 GB)...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.732778
20,1.127028
30,1.092619
40,1.106588
50,1.059095
60,1.121688


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


✅ Training completed in 411.50 seconds!
📊 Peak reserved VRAM: 8.34 GB (57.3% of total)


## 8. Inference After Fine-Tuning
Now let's evaluate our fine-tuned model on the same prompt using `FastLanguageModel.for_inference`.


In [32]:
# Prepare model for fast inference
FastLanguageModel.for_inference(model)

prompt = f"<|begin_of_text|><|start_header_id|>system<|end_header_id|>\n\nআপনি একজন বাংলাদেশি পারিবারিক আইন সহায়তা সহকারী। ব্যবহারকারীর সমস্যার ভিত্তিতে প্রাসঙ্গিক পরবর্তী প্রশ্ন করুন।<|eot_id|><|start_header_id|>user<|end_header_id|>\n\n{test_instruction}<|eot_id|><|start_header_id|>assistant<|end_header_id|>\n\n"

inputs = tokenizer([prompt], return_tensors="pt").to("cuda")
text_streamer = TextStreamer(tokenizer)

print("--- Output AFTER Fine-Tuning ---")
_ = model.generate(**inputs, streamer=text_streamer, max_new_tokens=512, use_cache=True)


--- Output AFTER Fine-Tuning ---
<|begin_of_text|><|begin_of_text|><|start_header_id|>system<|end_header_id|>

আপনি একজন বাংলাদেশি পারিবারিক আইন সহায়তা সহকারী। ব্যবহারকারীর সমস্যার ভিত্তিতে প্রাসঙ্গিক পরবর্তী প্রশ্ন করুন।<|eot_id|><|start_header_id|>user<|end_header_id|>

আমার স্বামী আমাকে দেনমোহরের টাকা পরিশোধ করতে অস্বীকার করছে। আমি কি আইনি পদক্ষেপ নিতে পারি?<|eot_id|><|start_header_id|>assistant<|end_header_id|>



Both `max_new_tokens` (=512) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


আপনি স্বামী আপনাকে দেনমোহরের টাকা পরিশোধ করতে অস্বীকার করছে তাহলে আপনি আপনার স্বামীকে সংশয়ে করতে পারেন। আপনি আপনার স্বামীকে সংশয়ে করতে হলে আপনি আপনার পারিবারিক আইন অফিসে অথবা আপনার স্বামীকে আইনি পদক্ষেপ নিতে পারেন।<|eot_id|>


## 9. Local Model Saving & Multi-Format Export
Choose the export option best suited for your local deployment target:

1. **LoRA Adapters** (~50-100 MB): Save lightweight adapter weights locally.
2. **Merged 16-bit Model**: Standalone full model for local servers (vLLM, Ollama, Triton).
3. **GGUF Quantization**: For ultra-fast local inference on CPU/GPU via Ollama, LM Studio, or llama.cpp.


In [33]:
adapter_output_dir = "/content/drive/MyDrive/lora_model_8b"

In [34]:
# Option 1: Save LoRA Adapters locally
model.save_pretrained(adapter_output_dir)
tokenizer.save_pretrained(adapter_output_dir)
print(f"✅ Saved LoRA adapters locally to ./{adapter_output_dir}")


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/lora_model_8b/tokenizer_config.json.


✅ Saved LoRA adapters locally to .//content/drive/MyDrive/lora_model_8b


In [35]:
# Option 2: Merge LoRA with base model and save full 16-bit weights locally
# Uncomment to execute:
merged_output_dir = "/content/drive/MyDrive/merged_16bit_model_8b"
model.save_pretrained_merged(merged_output_dir, tokenizer, save_method="merged_16bit")
print(f"✅ Saved merged 16-bit model to ./{merged_output_dir}")


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/merged_16bit_model_8b/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [05:32<00:00, 83.13s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)


Unsloth: Merging weights into 16bit:  25%|██▌       | 1/4 [05:03<15:10, 303.51s/it]


KeyboardInterrupt: 

In [ ]:
# Option 3: Export to GGUF format for Ollama / LM Studio
# Supported quantization methods: "q4_k_m", "q8_0", "f16"
gguf_output_dir = "/content/drive/MyDrive/llama_8b_gguf"
model.save_pretrained_gguf(gguf_output_dir, tokenizer, quantization_method="q4_k_m")
print(f"✅ Exported GGUF to ./{gguf_output_dir}")


Unsloth: ##### The current model auto adds a BOS token.
Unsloth: ##### Your chat template has a BOS token. We shall remove it temporarily.


Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/llama_8b_gguf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...
Cache check failed: model-00001-of-00004.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Preparing safetensor model files:   0%|          | 0/4 [00:00<?, ?it/s]

## 🎯 Next Steps & Deploying to Ollama
To run your exported GGUF model locally in **Ollama**:
1. Locate your exported `.gguf` file in `llama_1b_gguf/`.
2. Create a `Modelfile`:
   ```dockerfile
   FROM ./llama_1b_gguf/unsloth.Q4_K_M.gguf
   PARAMETER temperature 0.7
   PARAMETER top_p 0.9
   TEMPLATE """<|begin_of_text|><|start_header_id|>system<|end_header_id|>
   {{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>
   {{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>
   {{ .Response }}<|eot_id|>"""
   ```
3. Run:
   ```bash
   ollama create my-custom-llama -f Modelfile
   ollama run my-custom-llama
   ```
🎉 **All set! Happy Local Fine-Tuning!**
